data handling etl for the heathcare system glue

In [0]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F

## @params: [JOB_NAME]
args = getResolvedOptions(sys.argv, ['JOB_NAME'])

sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)


database = "health_monitor_db"

vitals_df = glueContext.create_dynamic_frame.from_catalog(
    database=database,
    table_name="patient_vitals"
).toDF()

patients_df = glueContext.create_dynamic_frame.from_catalog(
    database=database,
    table_name="patient_details"
).toDF()

devices_df = glueContext.create_dynamic_frame.from_catalog(
    database=database,
    table_name="device_reports"
).toDF()



# Remove duplicate records
vitals_df = vitals_df.dropDuplicates()

# Remove records with missing patient_id
vitals_df = vitals_df.filter(F.col("patient_id").isNotNull())

# Convert datatypes
vitals_df = (
    vitals_df
    .withColumn("heart_rate", F.col("heart_rate").cast("int"))
    .withColumn("oxygen_level", F.col("oxygen_level").cast("int"))
    .withColumn("temperature", F.col("temperature").cast("double"))
    .withColumn("timestamp", F.to_timestamp("timestamp"))
)

# Handle null readings
vitals_df = vitals_df.fillna({
    "heart_rate": 0,
    "oxygen_level": 0,
    "temperature": 0
})

# Remove incorrect values
vitals_df = vitals_df.filter(
    (F.col("heart_rate") >= 0) &
    (F.col("heart_rate") <= 250) &
    (F.col("oxygen_level") >= 0) &
    (F.col("oxygen_level") <= 100) &
    (F.col("temperature") >= 90) &
    (F.col("temperature") <= 110)
)



# Remove duplicate patient records
patients_df = patients_df.dropDuplicates(["patient_id"])

# Remove records with missing patient_id
patients_df = patients_df.filter(
    F.col("patient_id").isNotNull()
)

# Handle missing patient information
patients_df = patients_df.fillna({
    "name": "Unknown",
    "disease": "Unknown",
    "city": "Unknown",
    "gender": "Unknown"
})

# Remove incorrect ages
patients_df = patients_df.filter(
    (F.col("age") >= 0) &
    (F.col("age") <= 120)
)



# Remove duplicate device records
devices_df = devices_df.dropDuplicates()

# Remove records with missing patient_id
devices_df = devices_df.filter(
    F.col("patient_id").isNotNull()
)

# Handle missing device information
devices_df = devices_df.fillna({
    "device_type": "Unknown",
    "status": "Unknown"
})

# Convert timestamp
devices_df = devices_df.withColumn(
    "last_sync",
    F.to_timestamp("last_sync")
)

# Keep valid statuses only
devices_df = devices_df.filter(
    F.col("status").isin("Active", "Inactive", "Unknown")
)

vitals_df = vitals_df.withColumn(
    "health_status",
    F.when(
        (F.col("heart_rate") == 0) |
        (F.col("oxygen_level") == 0) |
        (F.col("temperature") == 0),
        "Missing Vitals"
    )
    .when(F.col("oxygen_level") < 90, "Critical")
    .when(F.col("temperature") > 100, "Fever")
    .when(F.col("heart_rate") > 100, "High Risk")
    .otherwise("Normal")
)

vitals_df = vitals_df.withColumn(
    "alert_flag",
    F.when(F.col("health_status") == "Normal", "N").otherwise("Y")
)

vitals_df = vitals_df.withColumn(
    "monitoring_date",
    F.to_date("timestamp")
)

vitals_df = vitals_df.withColumn(
    "monitoring_hour",
    F.hour("timestamp")
)


final_df = vitals_df.alias("v") \
    .join(patients_df.alias("p"), "patient_id", "left") \
    .join(devices_df.alias("d"), "patient_id", "left")

# FINAL OUTPUT

report_df = final_df.select(
    F.col("patient_id"),
    F.col("name").alias("patient_name"),
    F.col("disease"),
    F.col("heart_rate"),
    F.col("oxygen_level"),
    F.col("temperature"),
    F.col("health_status"),
    F.col("device_type"),
    F.col("monitoring_date")
)

# WRITE TO S3
output_path = "s3://patient-health-monitor01/PROCESSED/finalreport/"

report_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(output_path)

report_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", "s3://patient-health-monitor01/crawleroutput/") \
    .saveAsTable("health_monitor_db.patient_health_report")
    
job.commit()
    
